# T06 + T07 — Cổng khả thi kỹ thuật trên Tesla T4

Notebook này chạy **cả hai task trong một phiên** để tiết kiệm quota GPU (30 giờ/tuần). Không chứa logic nào, chỉ lấy code, cài và gọi script.

**Notebook settings trước khi chạy:**

- Accelerator: **GPU T4 x2**
- Internet: **On**
- Data: attach dataset `unicorn1209/vihallulens` — T07 **bắt buộc** cần, vì phải chạy trên mẫu ISE-DSC01 dài nhất
- Add-ons → Secrets: `HF_TOKEN` (không bắt buộc)

Không cần biết Kaggle gắn dataset vào đâu: script tự dò `/kaggle/input` tìm file `vihallu_train.csv`.

Chạy hết từ trên xuống rồi copy output của **ô 3, ô 4, ô 5 và ô 6** dán vào PR.

In [ ]:
# Ô 1 — lấy code. Chạy lại được nhiều lần: nếu thư mục đã có thì kéo bản mới về,
# vì `git clone` vào thư mục đã tồn tại sẽ hỏng và ta lặng lẽ chạy tiếp bằng code cũ.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

In [ ]:
# Ô 2 — cài đặt. Không cài lại torch: image Kaggle đã có bản dựng theo đúng CUDA của máy.
!pip install -q --no-deps -e .
!pip install -q -U bitsandbytes accelerate transformers pytest

In [ ]:
# Ô 3 — kiểm tra môi trường. Phải xanh trước khi chạy tiếp. Copy output dán vào PR.
# Chạy bằng tiến trình riêng chứ không import trong kernel: `pip install -e .` ghi một file
# .pth mà Python chỉ đọc lúc khởi động, nên kernel đang chạy sẵn có thể không thấy gói.
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: đã nạp từ Kaggle Secrets")
except Exception:
    print("HF_TOKEN: không có, vẫn chạy được vì Qwen2.5 là mô hình mở")

get_ipython().system("python scripts/probe_env.py")

In [ ]:
# Ô 4 — kiểm tra trên CPU: hook có nhận được attn_weights không, toán lookback có đúng không.
# Ô này hỏng thì DỪNG, đừng đốt quota GPU. Copy output dán vào PR.
!python -m pytest tests/test_attention_hook.py tests/test_attention_math.py -q
!python scripts/probe_attention_hook.py --tiny

In [ ]:
# Ô 5 — T06. Copy output dán vào PR.
!python scripts/probe_load_model.py

In [ ]:
# Ô 6 — T07. Copy output dán vào PR.
!python scripts/probe_attention_hook.py

In [ ]:
# Ô 7 — chỉ chạy nếu ô 6 báo có lớp nan/inf.
# Qwen2.5 huấn luyện ở bfloat16; float16 có dải số hẹp hơn nhiều nên vài lớp có thể tràn
# thành inf, và softmax của inf ra nan. float32 không tràn nhưng tốn gấp đôi bộ nhớ.
# Mô hình đã tải sẵn trong phiên nên ô này chỉ mất khoảng 20 giây để nạp lại.
!python scripts/probe_attention_hook.py --compute-dtype float32

In [ ]:
# Ô 8 — so sánh hai kiểu số trên 20 mẫu trải từ ngắn tới dài, để chốt dùng float16 hay float32.
# Trả lời hai câu: có phải lúc nào cũng chỉ lớp 27 hỏng không, và các lớp còn sống ở float16
# có khớp float32 không. Khoảng 4-5 phút GPU. Copy toàn bộ output dán vào PR.
!python scripts/compare_dtypes.py --per-dataset 10

## Nếu ô 6 báo hết bộ nhớ

Đi theo bảng sáu nấc lùi ở mục 5 của `CLAUDE.md`, **theo thứ tự**, đừng nhảy cóc:

```python
# Nấc 1 — hạ ngân sách token. Đo ở T05: chỉ cắt thêm 1,09 % mẫu ISE-DSC01.
!python scripts/probe_attention_hook.py --max-context-tokens 2048

# Nấc 4 — lùi mô hình, cùng họ nên không đổi dòng code nào.
!python scripts/probe_attention_hook.py --model Qwen/Qwen2.5-3B-Instruct
```

Nấc 2 và nấc 3 cần sửa code, nấc 3 là đổi kiến trúc nên phải hỏi trước. Ghi lại nấc nào đã thử vào **Nhật ký chặn** cuối `TASKS.md`.